# Image Regression ResNet50 Feature Extraction

## Feature Extraction ResNet50

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import pandas as pd

In [ ]:
class ImageFeatureExtractor:

  def __init__(self, model_name='resnet18', pretrained=True, layer_to_extract='avgpool'):
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.model = self._load_model(model_name, pretrained)
    self.model.eval()
    self.model.to(self.device)
    self.layer_to_extract = layer_to_extract
    self.features = None
    self._register_hook()
    self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])


  def _load_model(self, model_name, pretrained):

    if model_name == 'resnet18':
        model = models.resnet18(pretrained=pretrained)

    elif model_name == 'resnet50':
        model = models.resnet50(pretrained=pretrained)

    elif model_name == 'vgg16':
        model = models.vgg16(pretrained=pretrained)

    elif model_name == 'alexnet':
        model = models.alexnet(pretrained=pretrained)

    else:
        raise ValueError(f"Model '{model_name}' not supported. Choose from 'resnet18', 'resnet50', 'vgg16', 'alexnet'.")

    # Remove the final classification layer
    if hasattr(model, 'fc'): # For ResNet, AlexNet
        model.fc = torch.nn.Identity()

    elif hasattr(model, 'classifier') and isinstance(model.classifier, torch.nn.modules.container.Sequential): # For VGG, AlexNet
        # Keep all but the last layer (which is the classifier)
        model.classifier = torch.nn.Sequential(*list(model.classifier.children())[:-1])

    return model



  def _hook_fn(self, module, input, output):
    """Hook function to capture the output of the specified layer."""
    self.features = output.detach().squeeze().cpu()


  def _register_hook(self):
    """Registers a forward hook to the specified layer of the model."""
    target_layer = None
    if self.layer_to_extract == 'avgpool' and hasattr(self.model, 'avgpool'):
        target_layer = self.model.avgpool

    elif self.layer_to_extract == 'fc' and hasattr(self.model, 'fc'): # Before Identity for ResNet
      # If fc is Identity, we might want features from the layer before it (avgpool)
      # This logic needs to be careful depending on the model's architecture
      if isinstance(self.model.fc, torch.nn.Identity):
          print("Warning: 'fc' layer is Identity. Extracting from 'avgpool' instead for ResNet-like models.")
          target_layer = self.model.avgpool

      else:
          target_layer = self.model.fc

    elif self.layer_to_extract == 'classifier' and hasattr(self.model, 'classifier'):
      # For models like VGG, AlexNet, the classifier is a Sequential.
      # We want the output of the layer *before* the final classification.
      if isinstance(self.model.classifier, torch.nn.modules.container.Sequential) and len(self.model.classifier) > 1:
          target_layer = self.model.classifier[-1] # This would be the last layer before Identity was set
          if isinstance(target_layer, torch.nn.Identity): # If classifier was replaced with Identity
              print("Warning: Classifier last layer is Identity. Trying to get previous layer if available.")
              if len(self.model.classifier) > 2:
                  target_layer = self.model.classifier[-2] # Get the layer before the last one
              else:
                  raise ValueError(f"Could not find a suitable layer to extract from 'classifier' for {self.layer_to_extract}.")
      else:
          target_layer = self.model.classifier

    if target_layer is None:
        raise ValueError(f"Layer '{self.layer_to_extract}' not found or suitable for extraction in the selected model.")

    target_layer.register_forward_hook(self._hook_fn)
    print(f"Hook registered on layer: {self.layer_to_extract}")


  def extract(self, image_path):
    """
    Extracts features from a single image.

    Args:
        image_path (str): Path to the image file.

    Returns:
        np.ndarray: A 1D numpy array representing the extracted features (tabular data).
    """
    img = Image.open(image_path).convert('RGB')
    img_tensor = self.transform(img).unsqueeze(0).to(self.device)

    with torch.no_grad():
        _ = self.model(img_tensor) # Forward pass will trigger the hook

    if self.features is None:
        raise RuntimeError("Features were not captured. Check layer_to_extract and model architecture.")

    # Reset features for the next extraction
    features_to_return = self.features.numpy()
    self.features = None
    return features_to_return

In [ ]:
# Initialize the feature extractor using ResNet18 -> can choose 'resnet50', 'vgg16', 'alexnet' and 'avgpool' or 'fc' as layer
extractor = ImageFeatureExtractor(model_name='resnet50', pretrained=True, layer_to_extract='avgpool')

# Loop the folder image
import os
directory = "/content/input_images"

list_for_df = []

# Iterate over files in directory
for name in os.listdir(directory):

  image_file = os.path.join(directory, name)
  features = extractor.extract(image_file)

  list_for_df.append(features)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Hook registered on layer: avgpool


In [ ]:
df = pd.DataFrame(list_for_df)
df

,0,1,2,3,4,5,6,7,8,9,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
0,0.246206,0.130929,0.368296,0.615310,0.669143,0.15334,0.750426,0.532497,0.206516,0.131822,...,0.150118,0.236211,0.323442,0.116950,1.440531,0.550567,0.149462,0.263804,0.086210,0.083924
1,0.671374,0.694075,1.879372,0.458351,0.741315,0.23621,0.787268,0.655396,0.314910,1.715699,...,0.082924,0.392878,0.142819,0.631126,0.849930,0.451548,0.400683,0.168624,0.433643,0.666325
